In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kingburrito666/shakespeare-plays/alllines.txt
/kaggle/input/datasets/kingburrito666/shakespeare-plays/Shakespeare_data.csv
/kaggle/input/datasets/kingburrito666/shakespeare-plays/william-shakespeare-black-silhouette.jpg


# Step 1 — Import Libraries

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Step 2 — Load Dataset

In [3]:
with open('/kaggle/input/datasets/kingburrito666/shakespeare-plays/alllines.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:500])

"ACT I"
"SCENE I. London. The palace."
"Enter KING HENRY, LORD JOHN OF LANCASTER, the EARL of WESTMORELAND, SIR WALTER BLUNT, and others"
"So shaken as we are, so wan with care,"
"Find we a time for frighted peace to pant,"
"And breathe short-winded accents of new broils"
"To be commenced in strands afar remote."
"No more the thirsty entrance of this soil"
"Shall daub her lips with her own children's blood,"
"Nor more shall trenching war channel her fields,"
"Nor bruise her flowerets with the ar


# Step 3 — Build Vocabulary

Character-level tokenizer.

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

print(vocab_size)
print(data[:100])

78
tensor([ 4, 24, 26, 43,  2, 32,  4,  1,  4, 42, 26, 28, 37, 28,  2, 32, 11,  2,
        35, 66, 65, 55, 66, 65, 11,  2, 43, 59, 56,  2, 67, 52, 63, 52, 54, 56,
        11,  4,  1,  4, 28, 65, 71, 56, 69,  2, 34, 32, 37, 30,  2, 31, 28, 37,
        41, 48,  9,  2, 35, 38, 41, 27,  2, 33, 38, 31, 37,  2, 38, 29,  2, 35,
        24, 37, 26, 24, 42, 43, 28, 41,  9,  2, 71, 59, 56,  2, 28, 24, 41, 35,
         2, 66, 57,  2, 46, 28, 42, 43, 36, 38])


# Step 4 — Train/Test Split

In [5]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

# Step 5 — Hyperparameters

In [6]:
batch_size = 32
block_size = 128
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

# Step 6 — Batch Function

In [7]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x, y = x.to(device), y.to(device)

    return x, y

# Step 7 — Single Attention Head

In [8]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B,T,C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2,-1) * (k.shape[-1] ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        v = self.value(x)

        out = wei @ v

        return out

# Step 8 — Multi-Head Attention

In [9]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(head_size * num_heads, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat([h(x) for h in self.heads], dim=-1)

        out = self.dropout(self.proj(out))

        return out

# Step 9 — Feed Forward Network

In [10]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Step 10 — Transformer Block

In [11]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)

        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        x = x + self.sa(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x

# Step 11 — GPT Model

In [12]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B*T, C)

            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

# Step 12 — Train Model

In [17]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)

            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [18]:
model = GPTLanguageModel().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

for iter in range(max_iters):

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if iter % 500 == 0:
        print(iter, loss.item())

# Generate text

context = torch.zeros((1,1), dtype=torch.long, device=device)

generated = model.generate(context, max_new_tokens=500)

print(decode(generated[0].tolist()))

0 4.522154808044434
500 2.325439453125
1000 2.1523165702819824
1500 1.9720039367675781
2000 1.913765549659729
2500 1.8874671459197998
3000 1.83047354221344
3500 1.825071096420288
4000 1.7815887928009033
4500 1.7685881853103638
	Enter Roma caunter all presate-bliced cape him:"
"Is fell have tise lasse to will miver"
"grow, be appray plucio's my leal, them, belly good with"
rim anot, and we is do quorstand!"
"Give lost you, spale strother, and dreare in all strutant."
"And I what me it beate's him."
"What, you the wife infait died:"
"Come applosains and he visorand udent: be away not hose"
"Sindamert the with thoughough you shalloh, an, scant, antis king?"
"As othe pleasen, and unfuses like hangle for pract,"
"And I havi


In [24]:
start_text = "Hello world:"

# Encode input text
context = torch.tensor([encode(start_text)], dtype=torch.long).to(device)

# Generate next tokens
generated = model.generate(context, max_new_tokens=100)

# Decode back to text
output = decode(generated[0].tolist())

print(output)

Hello world: at be seat, for tap,"
"My conot in therefore Engent in the kiy, Rompt, 'saterecaw wither"
"Is eyes 
